In [23]:
import pandas as pd
import optuna
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import os
import shutil
from huggingface_hub import hf_hub_download

# Dataset & Preprocessing

In [2]:
data_path = 'dataset_model/dataset_trainable_rev2.csv'

In [3]:
if os.path.exists(data_path):
    print(f"Dataset Raw lokal sudah ada di '{data_path}'.")
else:
    print(f"Dataset Raw lokal belum ada. Mengunduh dataset dari Hugging Face...")
    
    temp_downloaded_path = hf_hub_download(
        repo_id="vel740/dhaka-smart-waste-processed",
        filename="processed/dataset_trainable_rev2.csv",
        repo_type="dataset"
    )
    
    os.makedirs('dataset_model', exist_ok=True)
    shutil.copy(temp_downloaded_path, data_path)
    print(f"Dataset selesai diunduh dan tersimpan di '{data_path}'.")

df = pd.read_csv(data_path)
print(f"Total baris awal: {len(df):,} baris")

Dataset Raw lokal sudah ada di 'dataset_model/dataset_trainable_rev2.csv'.
Total baris awal: 15,650 baris


In [4]:
# Hitung selisih target dan lag ke-3 untuk mendeteksi event pickup (> 20% drop)
target_diff = df['target_next_hour'] - df['fill_level_percent']
velocity_3h = df['lag_2h'] - df['lag_3h']

# Hanya ambil fase akumulasi murni (bebas dari drop pengangkutan sampah)
clean_mask = (
    (target_diff > -20.0) & 
    (df['velocity_1h'] > -20.0) & 
    (df['velocity_2h'] > -20.0) & 
    (velocity_3h > -20.0)
)

df_clean = df[clean_mask].reset_index(drop=True)
print(f"Total baris setelah pembersihan event pickup: {len(df_clean):,} baris (Dibuang: {len(df) - len(df_clean)} baris)")

Total baris setelah pembersihan event pickup: 15,650 baris (Dibuang: 0 baris)


In [5]:
df.head(5)

,bin_index,timestamp_unix,fill_level_percent,lag_1h,lag_2h,lag_3h,lag_24h,lag_48h,is_weekend,velocity_1h,velocity_2h,hour_sin,hour_cos,day_sin,day_cos,target_next_hour
0,0,1735840800,67.709990,68.51879,66.02875,67.10285,42.061382,14.483168,0,-0.808800,2.490044,0.000000,1.000000,-0.433884,-0.900969,70.403190
1,0,1735844400,70.403190,67.70999,68.51879,66.02875,42.492905,15.378024,0,2.693199,-0.808800,0.258819,0.965926,-0.433884,-0.900969,69.584000
2,0,1735848000,69.584000,70.40319,67.70999,68.51879,44.316532,16.866007,0,-0.819191,2.693199,0.500000,0.866025,-0.433884,-0.900969,72.692300
3,0,1735851600,72.692300,69.58400,70.40319,67.70999,43.484684,18.462954,0,3.108299,-0.819191,0.707107,0.707107,-0.433884,-0.900969,68.450325
4,0,1735855200,68.450325,72.69230,69.58400,70.40319,44.515327,20.921236,0,-4.241974,3.108299,0.866025,0.500000,-0.433884,-0.900969,72.475070


In [6]:
df_clean.head(5)

,bin_index,timestamp_unix,fill_level_percent,lag_1h,lag_2h,lag_3h,lag_24h,lag_48h,is_weekend,velocity_1h,velocity_2h,hour_sin,hour_cos,day_sin,day_cos,target_next_hour
0,0,1735840800,67.709990,68.51879,66.02875,67.10285,42.061382,14.483168,0,-0.808800,2.490044,0.000000,1.000000,-0.433884,-0.900969,70.403190
1,0,1735844400,70.403190,67.70999,68.51879,66.02875,42.492905,15.378024,0,2.693199,-0.808800,0.258819,0.965926,-0.433884,-0.900969,69.584000
2,0,1735848000,69.584000,70.40319,67.70999,68.51879,44.316532,16.866007,0,-0.819191,2.693199,0.500000,0.866025,-0.433884,-0.900969,72.692300
3,0,1735851600,72.692300,69.58400,70.40319,67.70999,43.484684,18.462954,0,3.108299,-0.819191,0.707107,0.707107,-0.433884,-0.900969,68.450325
4,0,1735855200,68.450325,72.69230,69.58400,70.40319,44.515327,20.921236,0,-4.241974,3.108299,0.866025,0.500000,-0.433884,-0.900969,72.475070


In [7]:
features = [
    'fill_level_percent', 'lag_1h', 'lag_2h', 'lag_3h', 
    'lag_24h', 'lag_48h', 'is_weekend',
    'velocity_1h', 'velocity_2h', 
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos'
]
target = 'target_next_hour'

# Temporal Split
split_timestamp = df_clean['timestamp_unix'].quantile(0.8)
train_mask = df_clean['timestamp_unix'] <= split_timestamp
test_mask = df_clean['timestamp_unix'] > split_timestamp

X_train_raw = df_clean.loc[train_mask, features].values
y_train_raw = df_clean.loc[train_mask, target].values
X_test_raw = df_clean.loc[test_mask, features].values
y_test_raw = df_clean.loc[test_mask, target].values

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

In [9]:
def create_sequences(X, y, seq_length):
    xs, ys = [], []
    for i in range(len(X) - seq_length):
        xs.append(X[i:(i + seq_length)])
        ys.append(y[i + seq_length])
    return np.array(xs), np.array(ys)

SEQUENCE_LENGTH = 12 # Looking at the past 12 hours to predict the 13th
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_raw, SEQUENCE_LENGTH)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_raw, SEQUENCE_LENGTH)

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")

train_dataset = TensorDataset(torch.tensor(X_train_seq, dtype=torch.float32), 
                              torch.tensor(y_train_seq, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_test_seq, dtype=torch.float32), 
                             torch.tensor(y_test_seq, dtype=torch.float32))

Training on device: cuda


In [11]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False) 
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Baseline Model

In [12]:
class SmartBinLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(SmartBinLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # batch_first=True ensures input format is (Batch, Sequence, Features)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :]) 
        return out.squeeze() # Squeeze removes unnecessary dimensions

In [13]:
model = SmartBinLSTM(input_size=len(features), hidden_size=32, num_layers=1, output_size=1).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

EPOCHS = 50

print("\n=== STARTING PYTORCH LSTM TRAINING ===")
for epoch in range(EPOCHS):
    model.train()
    train_losses = []
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()         # Clear old gradients
        predictions = model(X_batch)  # Forward pass
        loss = criterion(predictions, y_batch)
        
        loss.backward()               # Backpropagation
        optimizer.step()              # Update weights
        
        train_losses.append(loss.item())
        
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] - MSE Loss: {np.mean(train_losses):.4f}")


=== STARTING PYTORCH LSTM TRAINING ===
Epoch [10/50] - MSE Loss: 61.8358
Epoch [20/50] - MSE Loss: 59.3891
Epoch [30/50] - MSE Loss: 58.1465
Epoch [40/50] - MSE Loss: 56.1833
Epoch [50/50] - MSE Loss: 54.7104


In [14]:
model.eval()
all_preds = []
all_targets = []

In [15]:
with torch.no_grad(): # Disable gradient calculation for faster evaluation
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

final_mae = mean_absolute_error(all_targets, all_preds)
final_rmse = np.sqrt(mean_squared_error(all_targets, all_preds))

print("\n=== HASIL EVALUASI BASELINE LSTM ===")
print(f"Overall MAE (Semua Kondisi): {final_mae:.4f}%")
print(f"Overall RMSE               : {final_rmse:.4f}%")
print(f"LightGBM MAE Baseline      : 1.6079%")
print(f"Difference                 : {final_mae - 1.6079:+.4f}%")


=== HASIL EVALUASI BASELINE LSTM ===
Overall MAE (Semua Kondisi): 3.8451%
Overall RMSE               : 8.7608%
LightGBM MAE Baseline      : 1.6079%
Difference                 : +2.2372%


# Model Tuning

In [24]:
class SmartBinLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout_rate):
        super(SmartBinLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # PyTorch requires num_layers > 1 to apply internal LSTM dropout
        lstm_drop = dropout_rate if num_layers > 1 else 0.0
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                            batch_first=True, dropout=lstm_drop)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :]) 
        return out.squeeze()

In [25]:
def objective(trial):
    # --- Define Hyperparameters for this trial ---
    seq_length = trial.suggest_int('seq_length', 3, 12)
    hidden_size = trial.suggest_categorical('hidden_size', [16, 32, 64, 128])
    num_layers = trial.suggest_int('num_layers', 1, 3)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    
    # --- Dynamically rebuild 3D Tensors based on seq_length ---
    X_tr_seq, y_tr_seq = create_sequences(X_train_scaled, y_train_raw, seq_length)
    X_te_seq, y_te_seq = create_sequences(X_test_scaled, y_test_raw, seq_length)
    
    train_dataset = TensorDataset(torch.tensor(X_tr_seq, dtype=torch.float32), 
                                  torch.tensor(y_tr_seq, dtype=torch.float32))
    test_dataset = TensorDataset(torch.tensor(X_te_seq, dtype=torch.float32), 
                                 torch.tensor(y_te_seq, dtype=torch.float32))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # --- Initialize Model ---
    model = SmartBinLSTM(len(features), hidden_size, num_layers, 1, dropout_rate).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # --- Fast Training Loop (30 Epochs per trial for speed) ---
    EPOCHS = 30
    model.train()
    for epoch in range(EPOCHS):
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            
    # --- Evaluation ---
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(y_batch.numpy())
            
    # Optuna will attempt to minimize this MAE
    return mean_absolute_error(all_targets, all_preds)

In [26]:
study_name = "pytorch_lstm_smartbin_v1"
db_filename = "sqlite:///optuna_smartbin_study.sqlite3"

study = optuna.create_study(
    study_name=study_name,
    storage=db_filename,
    direction='minimize',
    load_if_exists=True
)

print(f"Starting GPU Optuna Tuning on {device}...")
# Running 50 trials. With your 5070 Ti, this should take less than 15 minutes.
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n=== OPTUNA LSTM TUNING COMPLETE ===")
print(f"Best Validation MAE: {study.best_value:.4f}%")
print("Best Parameters:", study.best_params)

[I 2026-11-01 20:21:57,908] A new study created in RDB with name: pytorch_lstm_smartbin_v1


Starting GPU Optuna Tuning on cuda...


Best trial: 0. Best value: 31.6388:   2%|▏         | 1/50 [00:07<05:59,  7.33s/it]

[I 2026-11-01 20:22:05,236] Trial 0 finished with value: 31.638797151017513 and parameters: {'seq_length': 7, 'hidden_size': 16, 'num_layers': 3, 'dropout_rate': 0.18907426851916692, 'learning_rate': 0.00016039772271470566, 'batch_size': 128}. Best is trial 0 with value: 31.638797151017513.


Best trial: 1. Best value: 22.7381:   4%|▍         | 2/50 [00:14<05:39,  7.07s/it]

...

[I 2026-11-01 20:29:46,022] Trial 47 finished with value: 6.157661421431436 and parameters: {'seq_length': 5, 'hidden_size': 64, 'num_layers': 1, 'dropout_rate': 0.27150395895631263, 'learning_rate': 0.0005022404905814929, 'batch_size': 128}. Best is trial 23 with value: 3.664345555390218.


Best trial: 23. Best value: 3.66435:  98%|█████████▊| 49/50 [08:05<00:10, 10.18s/it]

[I 2026-11-01 20:30:02,949] Trial 48 finished with value: 3.868141851825745 and parameters: {'seq_length': 6, 'hidden_size': 64, 'num_layers': 1, 'dropout_rate': 0.14658151654837165, 'learning_rate': 0.0012302321707430809, 'batch_size': 32}. Best is trial 23 with value: 3.664345555390218.


Best trial: 23. Best value: 3.66435: 100%|██████████| 50/50 [08:10<00:00,  9.80s/it]

[I 2026-11-01 20:30:07,940] Trial 49 finished with value: 4.0510708438418925 and parameters: {'seq_length': 3, 'hidden_size': 64, 'num_layers': 1, 'dropout_rate': 0.13640822089910978, 'learning_rate': 0.003066111779504549, 'batch_size': 128}. Best is trial 23 with value: 3.664345555390218.

=== OPTUNA LSTM TUNING COMPLETE ===
Best Validation MAE: 3.6643%
Best Parameters: {'seq_length': 6, 'hidden_size': 64, 'num_layers': 1, 'dropout_rate': 0.4077539707139816, 'learning_rate': 0.004607754398713653, 'batch_size': 64}


# Save Model

In [27]:
OPTIMAL_SEQ_LENGTH = 6
X_tr_final, y_tr_final = create_sequences(X_train_scaled, y_train_raw, OPTIMAL_SEQ_LENGTH)
X_te_final, y_te_final = create_sequences(X_test_scaled, y_test_raw, OPTIMAL_SEQ_LENGTH)

train_dataset_final = TensorDataset(torch.tensor(X_tr_final, dtype=torch.float32), 
                                    torch.tensor(y_tr_final, dtype=torch.float32))
# Using the best batch_size of 64
train_loader_final = DataLoader(train_dataset_final, batch_size=64, shuffle=False)

In [28]:
final_model = SmartBinLSTM(
    input_size=len(features), 
    hidden_size=64,           # Best hidden_size
    num_layers=1,             # Best num_layers
    output_size=1, 
    dropout_rate=0.4077       # Best dropout_rate
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=0.0046)

In [29]:
EPOCHS = 50
print("Training Final Golden Model on GPU...")
final_model.train()
for epoch in range(EPOCHS):
    for X_batch, y_batch in train_loader_final:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = final_model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

# 4. Safely Export the Model Weights
save_dir = 'model'
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, 'lstm_smartbin_16days_final.pth')

# We save the state_dict (the actual learned weights), not the entire class object
torch.save(final_model.state_dict(), model_path)
print(f"\nSUCCESS: Model weights safely locked and exported to -> {model_path}")

Training Final Golden Model on GPU...

SUCCESS: Model weights safely locked and exported to -> model\lstm_smartbin_16days_final.pth
